In [ ]:
import csv
import pandas as pd
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore') # Does not work for this kind of warning
pd.set_option('display.max_rows', None)


In [ ]:
# Introducing half clinic days

def generate_exact_40hr_schedule(start_date_str, total_weeks=5):
    
    workers = [
    "Dr. Alice Smith",
    "Nurse Bob Jones",
    "Dr. Charlie Brown",
    "Nurse Diana Prince",
    "Dr. Evan Wright",
    "Nurse Fiona Gallagher",
    "Dr. George Clark",
    "Nurse Hannah Abbott"
]
    if len(workers) != 8:
        raise ValueError(f" Roster balance requires exactly 8 workers. Found {len(workers)} names.")

    # Global tracking metrics over the 6 months
    global_counts = {
        w: {'Day_Shifts': 0, 'Night_Shifts': 0, 'Full_Clinic': 0, 'Half_Clinic': 0, 'Total_Hours': 0} 
        for w in workers
    }
    
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    schedule_data = []
    last_night_worker = None

    # Step-by-step weekly processing loops
    for week_idx in range(total_weeks):
        week_start_date = start_date + timedelta(weeks=week_idx)
        
        # Local trackers for the current 7-day operational block
        weekly_hospital_counts = {w: 0 for w in workers}
        week_days_manifest = {}
        
        # 1. Schedule 12-Hour Hospital Shifts first (Mon - Sun)
        for day_offset in range(7):
            current_date = week_start_date + timedelta(days=day_offset)
            date_str = current_date.strftime("%Y-%m-%d")
            day_name = current_date.strftime("%A")
            
            week_days_manifest[date_str] = {
                'day_name': day_name, 
                'day_hospital': None, 
                'night_hospital': None, 
                'full_clinic_staff': [],
                'half_clinic_staff': []
            }
            
            # --- Assign Day Shift ---
            available_for_day = [
                w for w in workers 
                if w != last_night_worker and weekly_hospital_counts[w] < 3
            ]
            available_for_day.sort(key=lambda w: (weekly_hospital_counts[w], global_counts[w]['Day_Shifts'], global_counts[w]['Total_Hours']))
            assigned_day_worker = available_for_day[0]
            
            week_days_manifest[date_str]['day_hospital'] = assigned_day_worker
            weekly_hospital_counts[assigned_day_worker] += 1
            global_counts[assigned_day_worker]['Day_Shifts'] += 1
            global_counts[assigned_day_worker]['Total_Hours'] += 12
            
            # --- Assign Night Shift ---
            available_for_night = [
                w for w in workers 
                if w != assigned_day_worker and w != last_night_worker and weekly_hospital_counts[w] < 3
            ]
            available_for_night.sort(key=lambda w: (weekly_hospital_counts[w], global_counts[w]['Night_Shifts'], global_counts[w]['Total_Hours']))
            assigned_night_worker = available_for_night[0]
            
            week_days_manifest[date_str]['night_hospital'] = assigned_night_worker
            weekly_hospital_counts[assigned_night_worker] += 1
            global_counts[assigned_night_worker]['Night_Shifts'] += 1
            global_counts[assigned_night_worker]['Total_Hours'] += 12
            
            last_night_worker = assigned_night_worker

        # 2. Backfill Clinic Requirements (Restricted to Weekdays Only)
        for worker in workers:
            shifts = weekly_hospital_counts[worker]
            
            # Determine the exact quota based on the 40-hour math matrix
            if shifts == 0:   req_full, req_half = 5, 0
            elif shifts == 1: req_full, req_half = 3, 1
            elif shifts == 2: req_full, req_half = 2, 0
            elif shifts == 3: req_full, req_half = 0, 1
            else:             req_full, req_half = 0, 0 # >3 shifts not allowed
            
            full_assigned = 0
            half_assigned = 0
            
            for date_str, day_data in week_days_manifest.items():
                # Stop if both quotas are full
                if full_assigned == req_full and half_assigned == req_half:
                    break
                
                # Weekday Check: Clinics are closed on weekends
                if day_data['day_name'] in ['Saturday', 'Sunday']:
                    continue
                
                # Conflict Checks
                is_on_day_hospital = day_data['day_hospital'] == worker
                is_on_night_hospital = day_data['night_hospital'] == worker
                
                prev_date = datetime.strptime(date_str, "%Y-%m-%d") - timedelta(days=1)
                prev_date_str = prev_date.strftime("%Y-%m-%d")
                was_resting = False
                if prev_date_str in week_days_manifest:
                    was_resting = week_days_manifest[prev_date_str]['night_hospital'] == worker
                
                # A worker can do clinic if they aren't on hospital duty and aren't post-night resting
                if not is_on_day_hospital and not is_on_night_hospital and not was_resting:
                    if full_assigned < req_full:
                        day_data['full_clinic_staff'].append(worker)
                        full_assigned += 1
                        global_counts[worker]['Full_Clinic'] += 1
                        global_counts[worker]['Total_Hours'] += 8
                    elif half_assigned < req_half:
                        day_data['half_clinic_staff'].append(worker)
                        half_assigned += 1
                        global_counts[worker]['Half_Clinic'] += 1
                        global_counts[worker]['Total_Hours'] += 4

        # 3. Format and save rows chronologically
        for date_str, day_data in sorted(week_days_manifest.items()):
            schedule_data.append({
                'Date': date_str,
                'Day of Week': day_data['day_name'],
                'Day Shift (12hr)': day_data['day_hospital'],
                'Night Shift (12hr)': day_data['night_hospital'],
                'Full Clinic (8hr)': ", ".join(day_data['full_clinic_staff']) if day_data['full_clinic_staff'] else "None",
                'Half Clinic (4hr)': ", ".join(day_data['half_clinic_staff']) if day_data['half_clinic_staff'] else "None"
            })

    # 4. Export to clean structural spreadsheet
    csv_filename = "hospital_exact_40hr_schedule.csv"
    fields = ['Date', 'Day of Week', 'Day Shift (12hr)', 'Night Shift (12hr)', 'Full Clinic (8hr)', 'Half Clinic (4hr)']
    with open(csv_filename, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(schedule_data)
        

    # Print the audited final metrics matrix
    print(f"Schedule successfully exported to {csv_filename}!\n")
    print("6-Month Cumulative Totals (Named 40 Hours/Week Equity Roster):")
    print(f"{'Worker Name':<22} | {'Day Shifts':<10} | {'Night Calls':<11} | {'Full Clinic':<11} | {'Half Clinic':<11} | {'Total Hours Logged':<18}")
    print("-" * 100)
    for worker, counts in sorted(global_counts.items()):
        print(f"{worker:<22} | {counts['Day_Shifts']:<10} | {counts['Night_Shifts']:<11} | {counts['Full_Clinic']:<11} | {counts['Half_Clinic']:<11} | {counts['Total_Hours']:<18} hrs")

# Run schedule engine starting tomorrow
generate_exact_40hr_schedule("2026-09-20")
